# Week 15 Learning Notebook

This notebook mirrors the daily lesson plan and includes runnable demo code cells.

In [1]:
from datetime import date
import warnings
import numpy as np
import pandas as pd

try:
    import yfinance as yf
except Exception:
    yf = None

try:
    from pandas_datareader import data as pdr
except Exception:
    pdr = None

np.set_printoptions(precision=4, suppress=True)
pd.options.display.float_format = '{:.6f}'.format

def _synthetic_price_frame(tickers, start='2018-01-01', periods=900, seed=7):
    idx = pd.bdate_range(start, periods=periods)
    out = pd.DataFrame(index=idx)
    rng = np.random.default_rng(seed)
    for i, ticker in enumerate(tickers):
        drift = 0.0002 + 0.00005 * (i + 1)
        vol = 0.010 + 0.002 * i
        rets = rng.normal(drift, vol, size=len(idx))
        out[ticker] = 100.0 * np.exp(np.cumsum(rets))
    return out

def load_market_prices(tickers, start='2018-01-01', end=None):
    end = end or date.today().isoformat()
    tickers = list(tickers)
    if yf is None:
        warnings.warn('yfinance unavailable, using synthetic data')
        return _synthetic_price_frame(tickers, start=start)

    try:
        raw = yf.download(tickers, start=start, end=end, auto_adjust=True, progress=False)
        if isinstance(raw.columns, pd.MultiIndex):
            if 'Close' in raw.columns.get_level_values(0):
                close = raw['Close']
            else:
                close = raw.xs('Close', axis=1, level=0, drop_level=True)
        else:
            close = raw.rename(columns={raw.columns[0]: tickers[0]})
        close = close.reindex(columns=tickers)
        close = close.dropna(how='all').ffill().dropna()
        if close.empty:
            raise ValueError('empty market data from Yahoo Finance')
        return close
    except Exception as exc:
        warnings.warn(f'Yahoo download failed ({exc}); using synthetic data')
        return _synthetic_price_frame(tickers, start=start)

def load_fred_series(series_id, start='2018-01-01', end=None):
    end = end or date.today().isoformat()
    if pdr is None:
        warnings.warn('pandas_datareader unavailable, using synthetic macro data')
        idx = pd.bdate_range(start, end)
        vals = np.linspace(1.0, 1.2, len(idx)) + np.sin(np.arange(len(idx)) / 25) * 0.02
        return pd.Series(vals, index=idx, name=series_id)

    try:
        ser = pdr.DataReader(series_id, 'fred', start, end)[series_id]
        ser = ser.ffill().dropna()
        if ser.empty:
            raise ValueError('empty FRED series')
        return ser
    except Exception as exc:
        warnings.warn(f'FRED download failed ({exc}); using synthetic macro data')
        idx = pd.bdate_range(start, end)
        vals = np.linspace(1.0, 1.2, len(idx)) + np.sin(np.arange(len(idx)) / 25) * 0.02
        return pd.Series(vals, index=idx, name=series_id)

# Week 15 Day 01: Option payoff structure

## Study Duration
- Planned effort: 4 hours

## 5-Block Daily Structure
- **Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Block 4 (60 min):** Python/pandas implementation and output verification.
- **Block 5 (30 min):** Practice questions, interview drill, and reflection.

## Why It Matters in Quant
Build options intuition for payoffs, Greeks, and hedging workflows.

## Continuity and Handoff
- Previous checkpoint: Week 14 Day 07: Portfolio Project
- Previous lesson file: content/week-14/day-07.md
- Today's deliverable: Implement payoff calculator for vanilla option strategies.
- Next handoff target: Week 15 Day 02: Greeks intuition
- Next lesson file: content/week-15/day-02.md

## Theory Concepts

### Concept 1: Calls and puts
Calls and puts is a core part of 'Derivatives and options basics'. Start with notation discipline: define weights, constraints, and risk units before solving the allocation problem. Then focus on allocation constraints, risk decomposition, and capital efficiency by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Long vs short payoff asymmetry
Long vs short payoff asymmetry is a core part of 'Derivatives and options basics'. Start with notation discipline: define weights, constraints, and risk units before solving the allocation problem. Then focus on allocation constraints, risk decomposition, and capital efficiency by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Moneyness intuition
Moneyness intuition is a core part of 'Derivatives and options basics'. Start with notation discipline: define weights, constraints, and risk units before solving the allocation problem. Then focus on allocation constraints, risk decomposition, and capital efficiency by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: Portfolio Return
$$
\mu_p=w^\top\mu
$$
**Plain-English interpretation:** Expected return from weighted assets.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

### Formula 2: Portfolio Variance
$$
\sigma_p^2=w^\top\Sigma w
$$
**Plain-English interpretation:** Quadratic risk engine.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

### Formula 3: Risk Contribution
$$
RC_i=w_i\frac{(\Sigma w)_i}{\sigma_p}
$$
**Plain-English interpretation:** Per-position risk budget.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $w$ | Portfolio weights | sum to 1 | [0.35,0.25,0.40] |
| $\Sigma$ | Covariance matrix | return^2 | 3x3 matrix |
| $D_{mod}$ | Modified duration | years | 5.8 |

## Real Trading Example
- Instruments: SPY, TLT, GLD, HYG
- Macro overlay (FRED): DGS10, T10YIE
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Plot payoff diagrams for basic option positions at expiry.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'Option payoff structure'.

## Step-by-Step Solved Problems
### Solved Problem 1: Portfolio expected return
Given:
- w=[0.6,0.4], mu=[0.12,0.08].
Solution:
1. $\mu_p=w^\top\mu$.
2. mu_p = 0.6*0.12 + 0.4*0.08 = 0.104.
Final answer: Portfolio expected return = 10.4%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: Portfolio volatility
Given:
- sigma1=0.20, sigma2=0.12, rho=0.30, w1=0.6, w2=0.4.
Solution:
1. $\sigma_p^2=w_1^2\sigma_1^2+w_2^2\sigma_2^2+2w_1w_2\rho\sigma_1\sigma_2$.
2. sigma_p^2 = 0.02048.
3. sigma_p = sqrt(0.02048) = 0.1431.
Final answer: Portfolio volatility = 14.31%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Duration shock
Given:
- Modified duration = 5.8, yield shift = +0.25%.
Solution:
1. $\Delta P/P\approx-D_{mod}\Delta y$.
2. DeltaP/P = -5.8*0.0025 = -0.0145.
Final answer: Approximate bond price change = -1.45%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 4: Position sizing with volatility guardrail
Given:
- Strategy annualized volatility estimate is 0.191.
- Portfolio risk budget target is 0.20.
- Position multiplier rule: scale = target_vol / strategy_vol, clipped to [0.20, 1.00].
Solution:
1. Compute raw scale = target_vol / strategy_vol.
2. raw_scale = 0.20/0.191 = 1.0471.
3. clipped_scale = min(1.00, max(0.20, 1.0471)) = 1.0000.
Final answer: Position multiplier = 1.0000.
Common trap: Ignoring volatility regime shifts and applying static position size in stressed markets.
Interpretation: State how this guardrail changes gross exposure before deployment.
## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Implement payoff calculator for vanilla option strategies.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
mu, cov = estimate_moments(asset_returns)
weights = solve_constrained_mv(mu, cov, max_weight=0.35)
risk_budget = risk_contributions(weights, cov)
rebalance_flag = should_rebalance(weights, target_weights, threshold=0.03)
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. PM interview question (Week 15 Day 01): Explain Portfolio Return and define every symbol clearly.
   - Model answer: "I use Portfolio Return to convert raw prices into a decision-ready metric. The formula is $\mu_p=w^\top\mu$. I define each symbol before computing it, verify units, and then interpret the output as a risk-adjusted trading input rather than a standalone signal."
2. Risk manager question: Using one real ticker from this lesson, what risk guardrail would you enforce?
   - Model answer: "I would run the metric on SPY and one higher-volatility asset, then enforce a volatility or drawdown cap. If the metric degrades in stressed regimes, I reduce gross exposure and require confirmation from a second risk check."
3. Production question: Why does 'Option payoff structure' matter in live trading systems?
   - Model answer: "Option payoff structure matters because it links model logic to real execution constraints. In production, I need reproducible calculations, explicit guardrails, and decision rules that stay stable when regime conditions change."

Scoring rubric:
- Full credit requires: correct notation, one numeric example, one explicit risk guardrail, and a production decision statement.

### Interview Drill
- Prompt: "Walk me through Option payoff structure as if you are presenting to a PM who cares about risk-adjusted returns."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision.
  3. Awareness of edge cases, costs, and failure modes.
- Model answer framework:
  - Context: define business objective and market regime.
  - Method: state formula and variables clearly.
  - Decision: explain one actionable rule and one risk guardrail.

## 2-Hour Extension Track (Required)

This section upgrades the day to a full 6-hour study model: 4-hour core lesson + 2-hour required extension.

- **Extension Block A (45 min):** Real-market case expansion.
  - Re-run today's workflow on one additional asset and one stress regime window.
  - Document one regime-specific failure mode and one mitigation rule.
- **Extension Block B (45 min):** Production-quality coding refinement.
  - Add one assertion for data integrity and one assertion for risk limits.
  - Save a short result table with assumptions, metric values, and decision rationale.
- **Extension Block C (30 min):** Interview simulation and review.
  - Deliver a 60-second PM pitch and a 60-second risk-manager response.
  - Include one numeric example from Week 15 Day 01 to prove notation fluency.

Extension completion checks:
- [ ] Stress-regime comparison completed
- [ ] Production guardrail assertions added and passed
- [ ] Interview simulation recorded with one numeric example
## Reflection Question
Which payoff shape best fits downside protection goals?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


## Week 15 Day 01 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [2]:
# Portfolio/risk demo: multi-asset allocation on real prices
np.random.seed(1501)
tickers = ['SPY', 'TLT', 'GLD', 'HYG']
prices = load_market_prices(tickers, start='2012-01-01')
returns = prices.pct_change().dropna()
mu = returns.mean().values * 252
cov = returns.cov().values * 252
vol = np.sqrt(np.diag(cov))

# inverse-volatility heuristic with normalization
inv_vol = 1 / np.maximum(vol, 1e-8)
weights = inv_vol / inv_vol.sum()
port_return = float(weights @ mu)
port_vol = float(np.sqrt(weights @ cov @ weights))
turnover_proxy = float(np.abs(np.diff(weights)).sum())

{
    'tickers': tickers,
    'weights': [float(round(x, 4)) for x in weights],
    'portfolio_expected_return': port_return,
    'portfolio_volatility': port_vol,
    'turnover_proxy': turnover_proxy,
}


{'tickers': ['SPY', 'TLT', 'GLD', 'HYG'],
 'weights': [0.1887, 0.2164, 0.1938, 0.4011],
 'portfolio_expected_return': 0.06894030109461007,
 'portfolio_volatility': 0.07641249751334192,
 'turnover_proxy': 0.25757139642322036}

## ReAct Verification Cell
Validate trade logic with a risk guardrail before reading the model quiz answers.

In [3]:
# ReAct-style verification: observe -> reason -> act -> verify
np.random.seed(10501)

observe_tickers = ['SPY', 'QQQ', 'TLT']
observe_prices = load_market_prices(observe_tickers, start='2020-01-01')
observe_returns = observe_prices.pct_change().dropna()

if observe_returns.empty:
    raise ValueError('No returns available for verification checks')

ann_vol = float(observe_returns['SPY'].std() * np.sqrt(252))
ann_ret = float((1 + observe_returns['SPY']).prod() ** (252 / len(observe_returns)) - 1)
sharpe_proxy = float((ann_ret - 0.03) / max(ann_vol, 1e-8))

# Risk-first deployment gate used in realistic interview responses.
guardrail = 'de-risk' if ann_vol > 0.30 else 'monitor'
decision = 'deploy-paper-trade' if sharpe_proxy > 0.40 and guardrail == 'monitor' else 'hold-and-review'

verification = {
    'topic': 'Option payoff structure',
    'week': 15,
    'day': 1,
    'observe_annual_return': ann_ret,
    'observe_annual_vol': ann_vol,
    'reason_sharpe_proxy': sharpe_proxy,
    'act_guardrail': guardrail,
    'verify_decision': decision,
}

verification


{'topic': 'Option payoff structure',
 'week': 15,
 'day': 1,
 'observe_annual_return': 0.14923378440213853,
 'observe_annual_vol': 0.20501289085899252,
 'reason_sharpe_proxy': 0.5815916448109953,
 'act_guardrail': 'monitor',
 'verify_decision': 'deploy-paper-trade'}

## Week 15 Day 01 Quiz

Topic: **Option payoff structure**

Real-world interview questions (answer first, then run the next cell for model answers):
1. PM question: Which formula from today's lesson directly drives a trade decision, and what does each symbol mean?
2. Risk question: Using one real ticker from today's example, what hard guardrail would you enforce before live deployment?
3. Communication question: In one minute, explain why this topic matters for production trading systems.

Scoring: full credit requires notation correctness, one numeric example, and one explicit risk control.

In [4]:
# Quiz model answers (auto-generated)
price_t_minus_1 = 116.000
price_t = 116.986
r_t = (price_t - price_t_minus_1) / price_t_minus_1
gross = 1 + r_t

print('Interview Question 1 (model answer):')
print('  I would use simple return to convert price moves into decision-ready percentages.')
print('  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)')
print('  P_(t-1):', price_t_minus_1)
print('  P_t    :', price_t)
print('  r_t    :', round(r_t, 6), '=>', f'{r_t*100:.2f}%')
print('  1+r_t  :', round(gross, 6))

print('\nInterview Question 2 (model answer):')
print('  For a real ticker like SPY, I would enforce a volatility guardrail: reduce position size')
print('  when realized volatility rises above regime threshold, then re-validate the signal logic.')

print('\nInterview Question 3 (model answer):')
print('  Topic:', 'Option payoff structure')
print('  This matters because production systems need reproducible metrics, explicit risk controls,')
print('  and clear decision rules that survive stressed market regimes.')

print('\nNumeric verification:')
print('  simple_return_expected =', 0.008500)
print('  gross_return_expected  =', 1.008500)


Interview Question 1 (model answer):
  I would use simple return to convert price moves into decision-ready percentages.
  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)
  P_(t-1): 116.0
  P_t    : 116.986
  r_t    : 0.0085 => 0.85%
  1+r_t  : 1.0085

Interview Question 2 (model answer):
  For a real ticker like SPY, I would enforce a volatility guardrail: reduce position size
  when realized volatility rises above regime threshold, then re-validate the signal logic.

Interview Question 3 (model answer):
  Topic: Option payoff structure
  This matters because production systems need reproducible metrics, explicit risk controls,
  and clear decision rules that survive stressed market regimes.

Numeric verification:
  simple_return_expected = 0.0085
  gross_return_expected  = 1.0085


# Week 15 Day 02: Greeks intuition

## Study Duration
- Planned effort: 4 hours

## 5-Block Daily Structure
- **Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Block 4 (60 min):** Python/pandas implementation and output verification.
- **Block 5 (30 min):** Practice questions, interview drill, and reflection.

## Why It Matters in Quant
Build options intuition for payoffs, Greeks, and hedging workflows.

## Continuity and Handoff
- Previous checkpoint: Week 15 Day 01: Option payoff structure
- Previous lesson file: content/week-15/day-01.md
- Today's deliverable: Create a Greeks approximation table for synthetic option quotes.
- Next handoff target: Week 15 Day 03: Volatility surface basics
- Next lesson file: content/week-15/day-03.md

## Theory Concepts

### Concept 1: Delta sensitivity
Delta sensitivity is a core part of 'Derivatives and options basics'. Start with notation discipline: define weights, constraints, and risk units before solving the allocation problem. Then focus on allocation constraints, risk decomposition, and capital efficiency by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Gamma curvature
Gamma curvature is a core part of 'Derivatives and options basics'. Start with notation discipline: define weights, constraints, and risk units before solving the allocation problem. Then focus on allocation constraints, risk decomposition, and capital efficiency by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Vega and implied volatility exposure
Vega and implied volatility exposure is a core part of 'Derivatives and options basics'. Start with notation discipline: define weights, constraints, and risk units before solving the allocation problem. Then focus on allocation constraints, risk decomposition, and capital efficiency by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: Portfolio Variance
$$
\sigma_p^2=w^\top\Sigma w
$$
**Plain-English interpretation:** Quadratic risk engine.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

### Formula 2: Risk Contribution
$$
RC_i=w_i\frac{(\Sigma w)_i}{\sigma_p}
$$
**Plain-English interpretation:** Per-position risk budget.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

### Formula 3: Duration Shock
$$
\frac{\Delta P}{P}\approx-D_{mod}\Delta y
$$
**Plain-English interpretation:** First-order bond sensitivity.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $w$ | Portfolio weights | sum to 1 | [0.35,0.25,0.40] |
| $\Sigma$ | Covariance matrix | return^2 | 3x3 matrix |
| $D_{mod}$ | Modified duration | years | 5.8 |

## Real Trading Example
- Instruments: SPY, TLT, GLD, HYG
- Macro overlay (FRED): DGS10, T10YIE
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Track delta and gamma behavior near at-the-money conditions.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'Greeks intuition'.

## Step-by-Step Solved Problems
### Solved Problem 1: Portfolio expected return
Given:
- w=[0.6,0.4], mu=[0.12,0.08].
Solution:
1. $\mu_p=w^\top\mu$.
2. mu_p = 0.6*0.12 + 0.4*0.08 = 0.104.
Final answer: Portfolio expected return = 10.4%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: Portfolio volatility
Given:
- sigma1=0.20, sigma2=0.12, rho=0.30, w1=0.6, w2=0.4.
Solution:
1. $\sigma_p^2=w_1^2\sigma_1^2+w_2^2\sigma_2^2+2w_1w_2\rho\sigma_1\sigma_2$.
2. sigma_p^2 = 0.02048.
3. sigma_p = sqrt(0.02048) = 0.1431.
Final answer: Portfolio volatility = 14.31%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Duration shock
Given:
- Modified duration = 5.8, yield shift = +0.25%.
Solution:
1. $\Delta P/P\approx-D_{mod}\Delta y$.
2. DeltaP/P = -5.8*0.0025 = -0.0145.
Final answer: Approximate bond price change = -1.45%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 4: Position sizing with volatility guardrail
Given:
- Strategy annualized volatility estimate is 0.192.
- Portfolio risk budget target is 0.20.
- Position multiplier rule: scale = target_vol / strategy_vol, clipped to [0.20, 1.00].
Solution:
1. Compute raw scale = target_vol / strategy_vol.
2. raw_scale = 0.20/0.192 = 1.0417.
3. clipped_scale = min(1.00, max(0.20, 1.0417)) = 1.0000.
Final answer: Position multiplier = 1.0000.
Common trap: Ignoring volatility regime shifts and applying static position size in stressed markets.
Interpretation: State how this guardrail changes gross exposure before deployment.
## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Create a Greeks approximation table for synthetic option quotes.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
mu, cov = estimate_moments(asset_returns)
weights = solve_constrained_mv(mu, cov, max_weight=0.35)
risk_budget = risk_contributions(weights, cov)
rebalance_flag = should_rebalance(weights, target_weights, threshold=0.03)
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. PM interview question (Week 15 Day 02): Explain Portfolio Variance and define every symbol clearly.
   - Model answer: "I use Portfolio Variance to convert raw prices into a decision-ready metric. The formula is $\sigma_p^2=w^\top\Sigma w$. I define each symbol before computing it, verify units, and then interpret the output as a risk-adjusted trading input rather than a standalone signal."
2. Risk manager question: Using one real ticker from this lesson, what risk guardrail would you enforce?
   - Model answer: "I would run the metric on SPY and one higher-volatility asset, then enforce a volatility or drawdown cap. If the metric degrades in stressed regimes, I reduce gross exposure and require confirmation from a second risk check."
3. Production question: Why does 'Greeks intuition' matter in live trading systems?
   - Model answer: "Greeks intuition matters because it links model logic to real execution constraints. In production, I need reproducible calculations, explicit guardrails, and decision rules that stay stable when regime conditions change."

Scoring rubric:
- Full credit requires: correct notation, one numeric example, one explicit risk guardrail, and a production decision statement.

### Interview Drill
- Prompt: "Walk me through Greeks intuition as if you are presenting to a PM who cares about risk-adjusted returns."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision.
  3. Awareness of edge cases, costs, and failure modes.
- Model answer framework:
  - Context: define business objective and market regime.
  - Method: state formula and variables clearly.
  - Decision: explain one actionable rule and one risk guardrail.

## 2-Hour Extension Track (Required)

This section upgrades the day to a full 6-hour study model: 4-hour core lesson + 2-hour required extension.

- **Extension Block A (45 min):** Real-market case expansion.
  - Re-run today's workflow on one additional asset and one stress regime window.
  - Document one regime-specific failure mode and one mitigation rule.
- **Extension Block B (45 min):** Production-quality coding refinement.
  - Add one assertion for data integrity and one assertion for risk limits.
  - Save a short result table with assumptions, metric values, and decision rationale.
- **Extension Block C (30 min):** Interview simulation and review.
  - Deliver a 60-second PM pitch and a 60-second risk-manager response.
  - Include one numeric example from Week 15 Day 02 to prove notation fluency.

Extension completion checks:
- [ ] Stress-regime comparison completed
- [ ] Production guardrail assertions added and passed
- [ ] Interview simulation recorded with one numeric example
## Reflection Question
How does gamma risk change near expiry?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


## Week 15 Day 02 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [5]:
# Portfolio/risk demo: multi-asset allocation on real prices
np.random.seed(1502)
tickers = ['SPY', 'TLT', 'GLD', 'HYG']
prices = load_market_prices(tickers, start='2012-01-01')
returns = prices.pct_change().dropna()
mu = returns.mean().values * 252
cov = returns.cov().values * 252
vol = np.sqrt(np.diag(cov))

# inverse-volatility heuristic with normalization
inv_vol = 1 / np.maximum(vol, 1e-8)
weights = inv_vol / inv_vol.sum()
port_return = float(weights @ mu)
port_vol = float(np.sqrt(weights @ cov @ weights))
turnover_proxy = float(np.abs(np.diff(weights)).sum())

{
    'tickers': tickers,
    'weights': [float(round(x, 4)) for x in weights],
    'portfolio_expected_return': port_return,
    'portfolio_volatility': port_vol,
    'turnover_proxy': turnover_proxy,
}


{'tickers': ['SPY', 'TLT', 'GLD', 'HYG'],
 'weights': [0.1887, 0.2164, 0.1938, 0.4011],
 'portfolio_expected_return': 0.06894030109461007,
 'portfolio_volatility': 0.07641249751334192,
 'turnover_proxy': 0.25757139642322036}

## ReAct Verification Cell
Validate trade logic with a risk guardrail before reading the model quiz answers.

In [6]:
# ReAct-style verification: observe -> reason -> act -> verify
np.random.seed(10502)

observe_tickers = ['SPY', 'QQQ', 'TLT']
observe_prices = load_market_prices(observe_tickers, start='2020-01-01')
observe_returns = observe_prices.pct_change().dropna()

if observe_returns.empty:
    raise ValueError('No returns available for verification checks')

ann_vol = float(observe_returns['SPY'].std() * np.sqrt(252))
ann_ret = float((1 + observe_returns['SPY']).prod() ** (252 / len(observe_returns)) - 1)
sharpe_proxy = float((ann_ret - 0.03) / max(ann_vol, 1e-8))

# Risk-first deployment gate used in realistic interview responses.
guardrail = 'de-risk' if ann_vol > 0.30 else 'monitor'
decision = 'deploy-paper-trade' if sharpe_proxy > 0.40 and guardrail == 'monitor' else 'hold-and-review'

verification = {
    'topic': 'Greeks intuition',
    'week': 15,
    'day': 2,
    'observe_annual_return': ann_ret,
    'observe_annual_vol': ann_vol,
    'reason_sharpe_proxy': sharpe_proxy,
    'act_guardrail': guardrail,
    'verify_decision': decision,
}

verification


{'topic': 'Greeks intuition',
 'week': 15,
 'day': 2,
 'observe_annual_return': 0.14923378440213853,
 'observe_annual_vol': 0.20501289085899252,
 'reason_sharpe_proxy': 0.5815916448109953,
 'act_guardrail': 'monitor',
 'verify_decision': 'deploy-paper-trade'}

## Week 15 Day 02 Quiz

Topic: **Greeks intuition**

Real-world interview questions (answer first, then run the next cell for model answers):
1. PM question: Which formula from today's lesson directly drives a trade decision, and what does each symbol mean?
2. Risk question: Using one real ticker from today's example, what hard guardrail would you enforce before live deployment?
3. Communication question: In one minute, explain why this topic matters for production trading systems.

Scoring: full credit requires notation correctness, one numeric example, and one explicit risk control.

In [7]:
# Quiz model answers (auto-generated)
price_t_minus_1 = 117.000
price_t = 118.053
r_t = (price_t - price_t_minus_1) / price_t_minus_1
gross = 1 + r_t

print('Interview Question 1 (model answer):')
print('  I would use simple return to convert price moves into decision-ready percentages.')
print('  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)')
print('  P_(t-1):', price_t_minus_1)
print('  P_t    :', price_t)
print('  r_t    :', round(r_t, 6), '=>', f'{r_t*100:.2f}%')
print('  1+r_t  :', round(gross, 6))

print('\nInterview Question 2 (model answer):')
print('  For a real ticker like SPY, I would enforce a volatility guardrail: reduce position size')
print('  when realized volatility rises above regime threshold, then re-validate the signal logic.')

print('\nInterview Question 3 (model answer):')
print('  Topic:', 'Greeks intuition')
print('  This matters because production systems need reproducible metrics, explicit risk controls,')
print('  and clear decision rules that survive stressed market regimes.')

print('\nNumeric verification:')
print('  simple_return_expected =', 0.009000)
print('  gross_return_expected  =', 1.009000)


Interview Question 1 (model answer):
  I would use simple return to convert price moves into decision-ready percentages.
  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)
  P_(t-1): 117.0
  P_t    : 118.053
  r_t    : 0.009 => 0.90%
  1+r_t  : 1.009

Interview Question 2 (model answer):
  For a real ticker like SPY, I would enforce a volatility guardrail: reduce position size
  when realized volatility rises above regime threshold, then re-validate the signal logic.

Interview Question 3 (model answer):
  Topic: Greeks intuition
  This matters because production systems need reproducible metrics, explicit risk controls,
  and clear decision rules that survive stressed market regimes.

Numeric verification:
  simple_return_expected = 0.009
  gross_return_expected  = 1.009


# Week 15 Day 03: Volatility surface basics

## Study Duration
- Planned effort: 4 hours

## 5-Block Daily Structure
- **Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Block 4 (60 min):** Python/pandas implementation and output verification.
- **Block 5 (30 min):** Practice questions, interview drill, and reflection.

## Why It Matters in Quant
Build options intuition for payoffs, Greeks, and hedging workflows.

## Continuity and Handoff
- Previous checkpoint: Week 15 Day 02: Greeks intuition
- Previous lesson file: content/week-15/day-02.md
- Today's deliverable: Build implied-vol surface toy visualization.
- Next handoff target: Week 15 Day 04: Hedging workflow basics
- Next lesson file: content/week-15/day-04.md

## Theory Concepts

### Concept 1: Implied volatility
Implied volatility is a core part of 'Derivatives and options basics'. Start with notation discipline: define weights, constraints, and risk units before solving the allocation problem. Then focus on allocation constraints, risk decomposition, and capital efficiency by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Smile/skew interpretation
Smile/skew interpretation is a core part of 'Derivatives and options basics'. Start with notation discipline: define weights, constraints, and risk units before solving the allocation problem. Then focus on allocation constraints, risk decomposition, and capital efficiency by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Vol regime context
Vol regime context is a core part of 'Derivatives and options basics'. Start with notation discipline: define weights, constraints, and risk units before solving the allocation problem. Then focus on allocation constraints, risk decomposition, and capital efficiency by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: Risk Contribution
$$
RC_i=w_i\frac{(\Sigma w)_i}{\sigma_p}
$$
**Plain-English interpretation:** Per-position risk budget.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

### Formula 2: Duration Shock
$$
\frac{\Delta P}{P}\approx-D_{mod}\Delta y
$$
**Plain-English interpretation:** First-order bond sensitivity.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

### Formula 3: CVaR
$$
CVaR_\alpha=E[L\mid L\ge VaR_\alpha]
$$
**Plain-English interpretation:** Tail-risk expectation.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $w$ | Portfolio weights | sum to 1 | [0.35,0.25,0.40] |
| $\Sigma$ | Covariance matrix | return^2 | 3x3 matrix |
| $D_{mod}$ | Modified duration | years | 5.8 |

## Real Trading Example
- Instruments: SPY, TLT, GLD, HYG
- Macro overlay (FRED): DGS10, T10YIE
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Simulate implied-volatility skew and discuss directional implications.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'Volatility surface basics'.

## Step-by-Step Solved Problems
### Solved Problem 1: Portfolio expected return
Given:
- w=[0.6,0.4], mu=[0.12,0.08].
Solution:
1. $\mu_p=w^\top\mu$.
2. mu_p = 0.6*0.12 + 0.4*0.08 = 0.104.
Final answer: Portfolio expected return = 10.4%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: Portfolio volatility
Given:
- sigma1=0.20, sigma2=0.12, rho=0.30, w1=0.6, w2=0.4.
Solution:
1. $\sigma_p^2=w_1^2\sigma_1^2+w_2^2\sigma_2^2+2w_1w_2\rho\sigma_1\sigma_2$.
2. sigma_p^2 = 0.02048.
3. sigma_p = sqrt(0.02048) = 0.1431.
Final answer: Portfolio volatility = 14.31%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Duration shock
Given:
- Modified duration = 5.8, yield shift = +0.25%.
Solution:
1. $\Delta P/P\approx-D_{mod}\Delta y$.
2. DeltaP/P = -5.8*0.0025 = -0.0145.
Final answer: Approximate bond price change = -1.45%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 4: Position sizing with volatility guardrail
Given:
- Strategy annualized volatility estimate is 0.193.
- Portfolio risk budget target is 0.20.
- Position multiplier rule: scale = target_vol / strategy_vol, clipped to [0.20, 1.00].
Solution:
1. Compute raw scale = target_vol / strategy_vol.
2. raw_scale = 0.20/0.193 = 1.0363.
3. clipped_scale = min(1.00, max(0.20, 1.0363)) = 1.0000.
Final answer: Position multiplier = 1.0000.
Common trap: Ignoring volatility regime shifts and applying static position size in stressed markets.
Interpretation: State how this guardrail changes gross exposure before deployment.
## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Build implied-vol surface toy visualization.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
mu, cov = estimate_moments(asset_returns)
weights = solve_constrained_mv(mu, cov, max_weight=0.35)
risk_budget = risk_contributions(weights, cov)
rebalance_flag = should_rebalance(weights, target_weights, threshold=0.03)
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. PM interview question (Week 15 Day 03): Explain Risk Contribution and define every symbol clearly.
   - Model answer: "I use Risk Contribution to convert raw prices into a decision-ready metric. The formula is $RC_i=w_i\frac{(\Sigma w)_i}{\sigma_p}$. I define each symbol before computing it, verify units, and then interpret the output as a risk-adjusted trading input rather than a standalone signal."
2. Risk manager question: Using one real ticker from this lesson, what risk guardrail would you enforce?
   - Model answer: "I would run the metric on SPY and one higher-volatility asset, then enforce a volatility or drawdown cap. If the metric degrades in stressed regimes, I reduce gross exposure and require confirmation from a second risk check."
3. Production question: Why does 'Volatility surface basics' matter in live trading systems?
   - Model answer: "Volatility surface basics matters because it links model logic to real execution constraints. In production, I need reproducible calculations, explicit guardrails, and decision rules that stay stable when regime conditions change."

Scoring rubric:
- Full credit requires: correct notation, one numeric example, one explicit risk guardrail, and a production decision statement.

### Interview Drill
- Prompt: "Walk me through Volatility surface basics as if you are presenting to a PM who cares about risk-adjusted returns."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision.
  3. Awareness of edge cases, costs, and failure modes.
- Model answer framework:
  - Context: define business objective and market regime.
  - Method: state formula and variables clearly.
  - Decision: explain one actionable rule and one risk guardrail.

## 2-Hour Extension Track (Required)

This section upgrades the day to a full 6-hour study model: 4-hour core lesson + 2-hour required extension.

- **Extension Block A (45 min):** Real-market case expansion.
  - Re-run today's workflow on one additional asset and one stress regime window.
  - Document one regime-specific failure mode and one mitigation rule.
- **Extension Block B (45 min):** Production-quality coding refinement.
  - Add one assertion for data integrity and one assertion for risk limits.
  - Save a short result table with assumptions, metric values, and decision rationale.
- **Extension Block C (30 min):** Interview simulation and review.
  - Deliver a 60-second PM pitch and a 60-second risk-manager response.
  - Include one numeric example from Week 15 Day 03 to prove notation fluency.

Extension completion checks:
- [ ] Stress-regime comparison completed
- [ ] Production guardrail assertions added and passed
- [ ] Interview simulation recorded with one numeric example
## Reflection Question
Why does skew differ across equities and indices?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


## Week 15 Day 03 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [8]:
# Portfolio/risk demo: multi-asset allocation on real prices
np.random.seed(1503)
tickers = ['SPY', 'TLT', 'GLD', 'HYG']
prices = load_market_prices(tickers, start='2012-01-01')
returns = prices.pct_change().dropna()
mu = returns.mean().values * 252
cov = returns.cov().values * 252
vol = np.sqrt(np.diag(cov))

# inverse-volatility heuristic with normalization
inv_vol = 1 / np.maximum(vol, 1e-8)
weights = inv_vol / inv_vol.sum()
port_return = float(weights @ mu)
port_vol = float(np.sqrt(weights @ cov @ weights))
turnover_proxy = float(np.abs(np.diff(weights)).sum())

{
    'tickers': tickers,
    'weights': [float(round(x, 4)) for x in weights],
    'portfolio_expected_return': port_return,
    'portfolio_volatility': port_vol,
    'turnover_proxy': turnover_proxy,
}


{'tickers': ['SPY', 'TLT', 'GLD', 'HYG'],
 'weights': [0.1887, 0.2164, 0.1938, 0.4011],
 'portfolio_expected_return': 0.06894030109461007,
 'portfolio_volatility': 0.07641249751334192,
 'turnover_proxy': 0.25757139642322036}

## ReAct Verification Cell
Validate trade logic with a risk guardrail before reading the model quiz answers.

In [9]:
# ReAct-style verification: observe -> reason -> act -> verify
np.random.seed(10503)

observe_tickers = ['SPY', 'QQQ', 'TLT']
observe_prices = load_market_prices(observe_tickers, start='2020-01-01')
observe_returns = observe_prices.pct_change().dropna()

if observe_returns.empty:
    raise ValueError('No returns available for verification checks')

ann_vol = float(observe_returns['SPY'].std() * np.sqrt(252))
ann_ret = float((1 + observe_returns['SPY']).prod() ** (252 / len(observe_returns)) - 1)
sharpe_proxy = float((ann_ret - 0.03) / max(ann_vol, 1e-8))

# Risk-first deployment gate used in realistic interview responses.
guardrail = 'de-risk' if ann_vol > 0.30 else 'monitor'
decision = 'deploy-paper-trade' if sharpe_proxy > 0.40 and guardrail == 'monitor' else 'hold-and-review'

verification = {
    'topic': 'Volatility surface basics',
    'week': 15,
    'day': 3,
    'observe_annual_return': ann_ret,
    'observe_annual_vol': ann_vol,
    'reason_sharpe_proxy': sharpe_proxy,
    'act_guardrail': guardrail,
    'verify_decision': decision,
}

verification


{'topic': 'Volatility surface basics',
 'week': 15,
 'day': 3,
 'observe_annual_return': 0.14923378440213853,
 'observe_annual_vol': 0.20501289085899252,
 'reason_sharpe_proxy': 0.5815916448109953,
 'act_guardrail': 'monitor',
 'verify_decision': 'deploy-paper-trade'}

## Week 15 Day 03 Quiz

Topic: **Volatility surface basics**

Real-world interview questions (answer first, then run the next cell for model answers):
1. PM question: Which formula from today's lesson directly drives a trade decision, and what does each symbol mean?
2. Risk question: Using one real ticker from today's example, what hard guardrail would you enforce before live deployment?
3. Communication question: In one minute, explain why this topic matters for production trading systems.

Scoring: full credit requires notation correctness, one numeric example, and one explicit risk control.

In [10]:
# Quiz model answers (auto-generated)
price_t_minus_1 = 118.000
price_t = 119.121
r_t = (price_t - price_t_minus_1) / price_t_minus_1
gross = 1 + r_t

print('Interview Question 1 (model answer):')
print('  I would use simple return to convert price moves into decision-ready percentages.')
print('  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)')
print('  P_(t-1):', price_t_minus_1)
print('  P_t    :', price_t)
print('  r_t    :', round(r_t, 6), '=>', f'{r_t*100:.2f}%')
print('  1+r_t  :', round(gross, 6))

print('\nInterview Question 2 (model answer):')
print('  For a real ticker like SPY, I would enforce a volatility guardrail: reduce position size')
print('  when realized volatility rises above regime threshold, then re-validate the signal logic.')

print('\nInterview Question 3 (model answer):')
print('  Topic:', 'Volatility surface basics')
print('  This matters because production systems need reproducible metrics, explicit risk controls,')
print('  and clear decision rules that survive stressed market regimes.')

print('\nNumeric verification:')
print('  simple_return_expected =', 0.009500)
print('  gross_return_expected  =', 1.009500)


Interview Question 1 (model answer):
  I would use simple return to convert price moves into decision-ready percentages.
  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)
  P_(t-1): 118.0
  P_t    : 119.121
  r_t    : 0.0095 => 0.95%
  1+r_t  : 1.0095

Interview Question 2 (model answer):
  For a real ticker like SPY, I would enforce a volatility guardrail: reduce position size
  when realized volatility rises above regime threshold, then re-validate the signal logic.

Interview Question 3 (model answer):
  Topic: Volatility surface basics
  This matters because production systems need reproducible metrics, explicit risk controls,
  and clear decision rules that survive stressed market regimes.

Numeric verification:
  simple_return_expected = 0.0095
  gross_return_expected  = 1.0095


# Week 15 Day 04: Hedging workflow basics

## Study Duration
- Planned effort: 4 hours

## 5-Block Daily Structure
- **Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Block 4 (60 min):** Python/pandas implementation and output verification.
- **Block 5 (30 min):** Practice questions, interview drill, and reflection.

## Why It Matters in Quant
Build options intuition for payoffs, Greeks, and hedging workflows.

## Continuity and Handoff
- Previous checkpoint: Week 15 Day 03: Volatility surface basics
- Previous lesson file: content/week-15/day-03.md
- Today's deliverable: Implement discrete hedging simulation and PnL decomposition.
- Next handoff target: Week 15 Day 05: Options strategy framing
- Next lesson file: content/week-15/day-05.md

## Theory Concepts

### Concept 1: Delta hedging intuition
Delta hedging intuition is a core part of 'Derivatives and options basics'. Start with notation discipline: define weights, constraints, and risk units before solving the allocation problem. Then focus on allocation constraints, risk decomposition, and capital efficiency by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Rehedging frequency tradeoff
Rehedging frequency tradeoff is a core part of 'Derivatives and options basics'. Start with notation discipline: define weights, constraints, and risk units before solving the allocation problem. Then focus on allocation constraints, risk decomposition, and capital efficiency by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Hedging costs and slippage
Hedging costs and slippage is a core part of 'Derivatives and options basics'. Start with notation discipline: define weights, constraints, and risk units before solving the allocation problem. Then focus on allocation constraints, risk decomposition, and capital efficiency by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: Duration Shock
$$
\frac{\Delta P}{P}\approx-D_{mod}\Delta y
$$
**Plain-English interpretation:** First-order bond sensitivity.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

### Formula 2: CVaR
$$
CVaR_\alpha=E[L\mid L\ge VaR_\alpha]
$$
**Plain-English interpretation:** Tail-risk expectation.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

### Formula 3: Portfolio Return
$$
\mu_p=w^\top\mu
$$
**Plain-English interpretation:** Expected return from weighted assets.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $w$ | Portfolio weights | sum to 1 | [0.35,0.25,0.40] |
| $\Sigma$ | Covariance matrix | return^2 | 3x3 matrix |
| $D_{mod}$ | Modified duration | years | 5.8 |

## Real Trading Example
- Instruments: SPY, TLT, GLD, HYG
- Macro overlay (FRED): DGS10, T10YIE
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Simulate a simple delta-hedging loop on synthetic price paths.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'Hedging workflow basics'.

## Step-by-Step Solved Problems
### Solved Problem 1: Portfolio expected return
Given:
- w=[0.6,0.4], mu=[0.12,0.08].
Solution:
1. $\mu_p=w^\top\mu$.
2. mu_p = 0.6*0.12 + 0.4*0.08 = 0.104.
Final answer: Portfolio expected return = 10.4%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: Portfolio volatility
Given:
- sigma1=0.20, sigma2=0.12, rho=0.30, w1=0.6, w2=0.4.
Solution:
1. $\sigma_p^2=w_1^2\sigma_1^2+w_2^2\sigma_2^2+2w_1w_2\rho\sigma_1\sigma_2$.
2. sigma_p^2 = 0.02048.
3. sigma_p = sqrt(0.02048) = 0.1431.
Final answer: Portfolio volatility = 14.31%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Duration shock
Given:
- Modified duration = 5.8, yield shift = +0.25%.
Solution:
1. $\Delta P/P\approx-D_{mod}\Delta y$.
2. DeltaP/P = -5.8*0.0025 = -0.0145.
Final answer: Approximate bond price change = -1.45%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 4: Position sizing with volatility guardrail
Given:
- Strategy annualized volatility estimate is 0.194.
- Portfolio risk budget target is 0.20.
- Position multiplier rule: scale = target_vol / strategy_vol, clipped to [0.20, 1.00].
Solution:
1. Compute raw scale = target_vol / strategy_vol.
2. raw_scale = 0.20/0.194 = 1.0309.
3. clipped_scale = min(1.00, max(0.20, 1.0309)) = 1.0000.
Final answer: Position multiplier = 1.0000.
Common trap: Ignoring volatility regime shifts and applying static position size in stressed markets.
Interpretation: State how this guardrail changes gross exposure before deployment.
## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Implement discrete hedging simulation and PnL decomposition.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
mu, cov = estimate_moments(asset_returns)
weights = solve_constrained_mv(mu, cov, max_weight=0.35)
risk_budget = risk_contributions(weights, cov)
rebalance_flag = should_rebalance(weights, target_weights, threshold=0.03)
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. PM interview question (Week 15 Day 04): Explain Duration Shock and define every symbol clearly.
   - Model answer: "I use Duration Shock to convert raw prices into a decision-ready metric. The formula is $\frac{\Delta P}{P}\approx-D_{mod}\Delta y$. I define each symbol before computing it, verify units, and then interpret the output as a risk-adjusted trading input rather than a standalone signal."
2. Risk manager question: Using one real ticker from this lesson, what risk guardrail would you enforce?
   - Model answer: "I would run the metric on SPY and one higher-volatility asset, then enforce a volatility or drawdown cap. If the metric degrades in stressed regimes, I reduce gross exposure and require confirmation from a second risk check."
3. Production question: Why does 'Hedging workflow basics' matter in live trading systems?
   - Model answer: "Hedging workflow basics matters because it links model logic to real execution constraints. In production, I need reproducible calculations, explicit guardrails, and decision rules that stay stable when regime conditions change."

Scoring rubric:
- Full credit requires: correct notation, one numeric example, one explicit risk guardrail, and a production decision statement.

### Interview Drill
- Prompt: "Walk me through Hedging workflow basics as if you are presenting to a PM who cares about risk-adjusted returns."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision.
  3. Awareness of edge cases, costs, and failure modes.
- Model answer framework:
  - Context: define business objective and market regime.
  - Method: state formula and variables clearly.
  - Decision: explain one actionable rule and one risk guardrail.

## 2-Hour Extension Track (Required)

This section upgrades the day to a full 6-hour study model: 4-hour core lesson + 2-hour required extension.

- **Extension Block A (45 min):** Real-market case expansion.
  - Re-run today's workflow on one additional asset and one stress regime window.
  - Document one regime-specific failure mode and one mitigation rule.
- **Extension Block B (45 min):** Production-quality coding refinement.
  - Add one assertion for data integrity and one assertion for risk limits.
  - Save a short result table with assumptions, metric values, and decision rationale.
- **Extension Block C (30 min):** Interview simulation and review.
  - Deliver a 60-second PM pitch and a 60-second risk-manager response.
  - Include one numeric example from Week 15 Day 04 to prove notation fluency.

Extension completion checks:
- [ ] Stress-regime comparison completed
- [ ] Production guardrail assertions added and passed
- [ ] Interview simulation recorded with one numeric example
## Reflection Question
When can hedging itself create risk concentration?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


## Week 15 Day 04 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [11]:
# Portfolio/risk demo: multi-asset allocation on real prices
np.random.seed(1504)
tickers = ['SPY', 'TLT', 'GLD', 'HYG']
prices = load_market_prices(tickers, start='2012-01-01')
returns = prices.pct_change().dropna()
mu = returns.mean().values * 252
cov = returns.cov().values * 252
vol = np.sqrt(np.diag(cov))

# inverse-volatility heuristic with normalization
inv_vol = 1 / np.maximum(vol, 1e-8)
weights = inv_vol / inv_vol.sum()
port_return = float(weights @ mu)
port_vol = float(np.sqrt(weights @ cov @ weights))
turnover_proxy = float(np.abs(np.diff(weights)).sum())

{
    'tickers': tickers,
    'weights': [float(round(x, 4)) for x in weights],
    'portfolio_expected_return': port_return,
    'portfolio_volatility': port_vol,
    'turnover_proxy': turnover_proxy,
}


{'tickers': ['SPY', 'TLT', 'GLD', 'HYG'],
 'weights': [0.1887, 0.2164, 0.1938, 0.4011],
 'portfolio_expected_return': 0.06894030109461007,
 'portfolio_volatility': 0.07641249751334192,
 'turnover_proxy': 0.25757139642322036}

## ReAct Verification Cell
Validate trade logic with a risk guardrail before reading the model quiz answers.

In [12]:
# ReAct-style verification: observe -> reason -> act -> verify
np.random.seed(10504)

observe_tickers = ['SPY', 'QQQ', 'TLT']
observe_prices = load_market_prices(observe_tickers, start='2020-01-01')
observe_returns = observe_prices.pct_change().dropna()

if observe_returns.empty:
    raise ValueError('No returns available for verification checks')

ann_vol = float(observe_returns['SPY'].std() * np.sqrt(252))
ann_ret = float((1 + observe_returns['SPY']).prod() ** (252 / len(observe_returns)) - 1)
sharpe_proxy = float((ann_ret - 0.03) / max(ann_vol, 1e-8))

# Risk-first deployment gate used in realistic interview responses.
guardrail = 'de-risk' if ann_vol > 0.30 else 'monitor'
decision = 'deploy-paper-trade' if sharpe_proxy > 0.40 and guardrail == 'monitor' else 'hold-and-review'

verification = {
    'topic': 'Hedging workflow basics',
    'week': 15,
    'day': 4,
    'observe_annual_return': ann_ret,
    'observe_annual_vol': ann_vol,
    'reason_sharpe_proxy': sharpe_proxy,
    'act_guardrail': guardrail,
    'verify_decision': decision,
}

verification


{'topic': 'Hedging workflow basics',
 'week': 15,
 'day': 4,
 'observe_annual_return': 0.14923378440213853,
 'observe_annual_vol': 0.20501289085899252,
 'reason_sharpe_proxy': 0.5815916448109953,
 'act_guardrail': 'monitor',
 'verify_decision': 'deploy-paper-trade'}

## Week 15 Day 04 Quiz

Topic: **Hedging workflow basics**

Real-world interview questions (answer first, then run the next cell for model answers):
1. PM question: Which formula from today's lesson directly drives a trade decision, and what does each symbol mean?
2. Risk question: Using one real ticker from today's example, what hard guardrail would you enforce before live deployment?
3. Communication question: In one minute, explain why this topic matters for production trading systems.

Scoring: full credit requires notation correctness, one numeric example, and one explicit risk control.

In [13]:
# Quiz model answers (auto-generated)
price_t_minus_1 = 119.000
price_t = 120.190
r_t = (price_t - price_t_minus_1) / price_t_minus_1
gross = 1 + r_t

print('Interview Question 1 (model answer):')
print('  I would use simple return to convert price moves into decision-ready percentages.')
print('  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)')
print('  P_(t-1):', price_t_minus_1)
print('  P_t    :', price_t)
print('  r_t    :', round(r_t, 6), '=>', f'{r_t*100:.2f}%')
print('  1+r_t  :', round(gross, 6))

print('\nInterview Question 2 (model answer):')
print('  For a real ticker like SPY, I would enforce a volatility guardrail: reduce position size')
print('  when realized volatility rises above regime threshold, then re-validate the signal logic.')

print('\nInterview Question 3 (model answer):')
print('  Topic:', 'Hedging workflow basics')
print('  This matters because production systems need reproducible metrics, explicit risk controls,')
print('  and clear decision rules that survive stressed market regimes.')

print('\nNumeric verification:')
print('  simple_return_expected =', 0.010000)
print('  gross_return_expected  =', 1.010000)


Interview Question 1 (model answer):
  I would use simple return to convert price moves into decision-ready percentages.
  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)
  P_(t-1): 119.0
  P_t    : 120.19
  r_t    : 0.01 => 1.00%
  1+r_t  : 1.01

Interview Question 2 (model answer):
  For a real ticker like SPY, I would enforce a volatility guardrail: reduce position size
  when realized volatility rises above regime threshold, then re-validate the signal logic.

Interview Question 3 (model answer):
  Topic: Hedging workflow basics
  This matters because production systems need reproducible metrics, explicit risk controls,
  and clear decision rules that survive stressed market regimes.

Numeric verification:
  simple_return_expected = 0.01
  gross_return_expected  = 1.01


# Week 15 Day 05: Options strategy framing

## Study Duration
- Planned effort: 4 hours

## 5-Block Daily Structure
- **Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Block 4 (60 min):** Python/pandas implementation and output verification.
- **Block 5 (30 min):** Practice questions, interview drill, and reflection.

## Why It Matters in Quant
Build options intuition for payoffs, Greeks, and hedging workflows.

## Continuity and Handoff
- Previous checkpoint: Week 15 Day 04: Hedging workflow basics
- Previous lesson file: content/week-15/day-04.md
- Today's deliverable: Build strategy scenario matrix with payoff summaries.
- Next handoff target: Week 15 Day 06: Revision Sprint
- Next lesson file: content/week-15/day-06.md

## Theory Concepts

### Concept 1: Directional vs volatility bets
Directional vs volatility bets is a core part of 'Derivatives and options basics'. Start with notation discipline: define weights, constraints, and risk units before solving the allocation problem. Then focus on allocation constraints, risk decomposition, and capital efficiency by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Risk-defined structures
Risk-defined structures is a core part of 'Derivatives and options basics'. Start with notation discipline: define weights, constraints, and risk units before solving the allocation problem. Then focus on allocation constraints, risk decomposition, and capital efficiency by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Scenario-based comparison
Scenario-based comparison is a core part of 'Derivatives and options basics'. Start with notation discipline: define weights, constraints, and risk units before solving the allocation problem. Then focus on allocation constraints, risk decomposition, and capital efficiency by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: CVaR
$$
CVaR_\alpha=E[L\mid L\ge VaR_\alpha]
$$
**Plain-English interpretation:** Tail-risk expectation.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

### Formula 2: Portfolio Return
$$
\mu_p=w^\top\mu
$$
**Plain-English interpretation:** Expected return from weighted assets.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

### Formula 3: Portfolio Variance
$$
\sigma_p^2=w^\top\Sigma w
$$
**Plain-English interpretation:** Quadratic risk engine.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $w$ | Portfolio weights | sum to 1 | [0.35,0.25,0.40] |
| $\Sigma$ | Covariance matrix | return^2 | 3x3 matrix |
| $D_{mod}$ | Modified duration | years | 5.8 |

## Real Trading Example
- Instruments: SPY, TLT, GLD, HYG
- Macro overlay (FRED): DGS10, T10YIE
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Compare covered call, protective put, and straddle under scenarios.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'Options strategy framing'.

## Step-by-Step Solved Problems
### Solved Problem 1: Portfolio expected return
Given:
- w=[0.6,0.4], mu=[0.12,0.08].
Solution:
1. $\mu_p=w^\top\mu$.
2. mu_p = 0.6*0.12 + 0.4*0.08 = 0.104.
Final answer: Portfolio expected return = 10.4%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: Portfolio volatility
Given:
- sigma1=0.20, sigma2=0.12, rho=0.30, w1=0.6, w2=0.4.
Solution:
1. $\sigma_p^2=w_1^2\sigma_1^2+w_2^2\sigma_2^2+2w_1w_2\rho\sigma_1\sigma_2$.
2. sigma_p^2 = 0.02048.
3. sigma_p = sqrt(0.02048) = 0.1431.
Final answer: Portfolio volatility = 14.31%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Duration shock
Given:
- Modified duration = 5.8, yield shift = +0.25%.
Solution:
1. $\Delta P/P\approx-D_{mod}\Delta y$.
2. DeltaP/P = -5.8*0.0025 = -0.0145.
Final answer: Approximate bond price change = -1.45%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 4: Position sizing with volatility guardrail
Given:
- Strategy annualized volatility estimate is 0.195.
- Portfolio risk budget target is 0.20.
- Position multiplier rule: scale = target_vol / strategy_vol, clipped to [0.20, 1.00].
Solution:
1. Compute raw scale = target_vol / strategy_vol.
2. raw_scale = 0.20/0.195 = 1.0256.
3. clipped_scale = min(1.00, max(0.20, 1.0256)) = 1.0000.
Final answer: Position multiplier = 1.0000.
Common trap: Ignoring volatility regime shifts and applying static position size in stressed markets.
Interpretation: State how this guardrail changes gross exposure before deployment.
## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Build strategy scenario matrix with payoff summaries.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
mu, cov = estimate_moments(asset_returns)
weights = solve_constrained_mv(mu, cov, max_weight=0.35)
risk_budget = risk_contributions(weights, cov)
rebalance_flag = should_rebalance(weights, target_weights, threshold=0.03)
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. PM interview question (Week 15 Day 05): Explain CVaR and define every symbol clearly.
   - Model answer: "I use CVaR to convert raw prices into a decision-ready metric. The formula is $CVaR_\alpha=E[L\mid L\ge VaR_\alpha]$. I define each symbol before computing it, verify units, and then interpret the output as a risk-adjusted trading input rather than a standalone signal."
2. Risk manager question: Using one real ticker from this lesson, what risk guardrail would you enforce?
   - Model answer: "I would run the metric on SPY and one higher-volatility asset, then enforce a volatility or drawdown cap. If the metric degrades in stressed regimes, I reduce gross exposure and require confirmation from a second risk check."
3. Production question: Why does 'Options strategy framing' matter in live trading systems?
   - Model answer: "Options strategy framing matters because it links model logic to real execution constraints. In production, I need reproducible calculations, explicit guardrails, and decision rules that stay stable when regime conditions change."

Scoring rubric:
- Full credit requires: correct notation, one numeric example, one explicit risk guardrail, and a production decision statement.

### Interview Drill
- Prompt: "Walk me through Options strategy framing as if you are presenting to a PM who cares about risk-adjusted returns."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision.
  3. Awareness of edge cases, costs, and failure modes.
- Model answer framework:
  - Context: define business objective and market regime.
  - Method: state formula and variables clearly.
  - Decision: explain one actionable rule and one risk guardrail.

## 2-Hour Extension Track (Required)

This section upgrades the day to a full 6-hour study model: 4-hour core lesson + 2-hour required extension.

- **Extension Block A (45 min):** Real-market case expansion.
  - Re-run today's workflow on one additional asset and one stress regime window.
  - Document one regime-specific failure mode and one mitigation rule.
- **Extension Block B (45 min):** Production-quality coding refinement.
  - Add one assertion for data integrity and one assertion for risk limits.
  - Save a short result table with assumptions, metric values, and decision rationale.
- **Extension Block C (30 min):** Interview simulation and review.
  - Deliver a 60-second PM pitch and a 60-second risk-manager response.
  - Include one numeric example from Week 15 Day 05 to prove notation fluency.

Extension completion checks:
- [ ] Stress-regime comparison completed
- [ ] Production guardrail assertions added and passed
- [ ] Interview simulation recorded with one numeric example
## Reflection Question
Which strategy aligns with your current risk tolerance and why?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


## Week 15 Day 05 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [14]:
# Portfolio/risk demo: multi-asset allocation on real prices
np.random.seed(1505)
tickers = ['SPY', 'TLT', 'GLD', 'HYG']
prices = load_market_prices(tickers, start='2012-01-01')
returns = prices.pct_change().dropna()
mu = returns.mean().values * 252
cov = returns.cov().values * 252
vol = np.sqrt(np.diag(cov))

# inverse-volatility heuristic with normalization
inv_vol = 1 / np.maximum(vol, 1e-8)
weights = inv_vol / inv_vol.sum()
port_return = float(weights @ mu)
port_vol = float(np.sqrt(weights @ cov @ weights))
turnover_proxy = float(np.abs(np.diff(weights)).sum())

{
    'tickers': tickers,
    'weights': [float(round(x, 4)) for x in weights],
    'portfolio_expected_return': port_return,
    'portfolio_volatility': port_vol,
    'turnover_proxy': turnover_proxy,
}


{'tickers': ['SPY', 'TLT', 'GLD', 'HYG'],
 'weights': [0.1887, 0.2164, 0.1938, 0.4011],
 'portfolio_expected_return': 0.06894030109461007,
 'portfolio_volatility': 0.07641249751334192,
 'turnover_proxy': 0.25757139642322036}

## ReAct Verification Cell
Validate trade logic with a risk guardrail before reading the model quiz answers.

In [15]:
# ReAct-style verification: observe -> reason -> act -> verify
np.random.seed(10505)

observe_tickers = ['SPY', 'QQQ', 'TLT']
observe_prices = load_market_prices(observe_tickers, start='2020-01-01')
observe_returns = observe_prices.pct_change().dropna()

if observe_returns.empty:
    raise ValueError('No returns available for verification checks')

ann_vol = float(observe_returns['SPY'].std() * np.sqrt(252))
ann_ret = float((1 + observe_returns['SPY']).prod() ** (252 / len(observe_returns)) - 1)
sharpe_proxy = float((ann_ret - 0.03) / max(ann_vol, 1e-8))

# Risk-first deployment gate used in realistic interview responses.
guardrail = 'de-risk' if ann_vol > 0.30 else 'monitor'
decision = 'deploy-paper-trade' if sharpe_proxy > 0.40 and guardrail == 'monitor' else 'hold-and-review'

verification = {
    'topic': 'Options strategy framing',
    'week': 15,
    'day': 5,
    'observe_annual_return': ann_ret,
    'observe_annual_vol': ann_vol,
    'reason_sharpe_proxy': sharpe_proxy,
    'act_guardrail': guardrail,
    'verify_decision': decision,
}

verification


{'topic': 'Options strategy framing',
 'week': 15,
 'day': 5,
 'observe_annual_return': 0.14923378440213853,
 'observe_annual_vol': 0.20501289085899252,
 'reason_sharpe_proxy': 0.5815916448109953,
 'act_guardrail': 'monitor',
 'verify_decision': 'deploy-paper-trade'}

## Week 15 Day 05 Quiz

Topic: **Options strategy framing**

Real-world interview questions (answer first, then run the next cell for model answers):
1. PM question: Which formula from today's lesson directly drives a trade decision, and what does each symbol mean?
2. Risk question: Using one real ticker from today's example, what hard guardrail would you enforce before live deployment?
3. Communication question: In one minute, explain why this topic matters for production trading systems.

Scoring: full credit requires notation correctness, one numeric example, and one explicit risk control.

In [16]:
# Quiz model answers (auto-generated)
price_t_minus_1 = 120.000
price_t = 121.260
r_t = (price_t - price_t_minus_1) / price_t_minus_1
gross = 1 + r_t

print('Interview Question 1 (model answer):')
print('  I would use simple return to convert price moves into decision-ready percentages.')
print('  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)')
print('  P_(t-1):', price_t_minus_1)
print('  P_t    :', price_t)
print('  r_t    :', round(r_t, 6), '=>', f'{r_t*100:.2f}%')
print('  1+r_t  :', round(gross, 6))

print('\nInterview Question 2 (model answer):')
print('  For a real ticker like SPY, I would enforce a volatility guardrail: reduce position size')
print('  when realized volatility rises above regime threshold, then re-validate the signal logic.')

print('\nInterview Question 3 (model answer):')
print('  Topic:', 'Options strategy framing')
print('  This matters because production systems need reproducible metrics, explicit risk controls,')
print('  and clear decision rules that survive stressed market regimes.')

print('\nNumeric verification:')
print('  simple_return_expected =', 0.010500)
print('  gross_return_expected  =', 1.010500)


Interview Question 1 (model answer):
  I would use simple return to convert price moves into decision-ready percentages.
  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)
  P_(t-1): 120.0
  P_t    : 121.26
  r_t    : 0.0105 => 1.05%
  1+r_t  : 1.0105

Interview Question 2 (model answer):
  For a real ticker like SPY, I would enforce a volatility guardrail: reduce position size
  when realized volatility rises above regime threshold, then re-validate the signal logic.

Interview Question 3 (model answer):
  Topic: Options strategy framing
  This matters because production systems need reproducible metrics, explicit risk controls,
  and clear decision rules that survive stressed market regimes.

Numeric verification:
  simple_return_expected = 0.0105
  gross_return_expected  = 1.0105


# Week 15 Day 06: Revision Sprint

## Study Duration
- Planned effort: 2 hours

## Continuity and Handoff
- Previous checkpoint: Week 15 Day 05: Options strategy framing
- Previous lesson file: content/week-15/day-05.md
- Today's deliverable: Revision checklist with corrected errors and next-week retest priorities.
- Next handoff target: Week 15 Day 07: Portfolio Project
- Next lesson file: content/week-15/day-07.md

## Revision Plan
- 30 minutes: active recall of weekday concepts
- 40 minutes: rebuild one code workflow from memory
- 30 minutes: error log triage and retest plan
- 20 minutes: update progress tracker and notes

## Focus Areas
- Redraw key payoff and Greek intuition charts
- Re-run one hedging simulation
- Review assumptions in volatility-surface interpretation

## Revision Output
- [ ] Updated error log entries
- [ ] Weak concepts re-tested
- [ ] Next-week risk list prepared


## Week 15 Day 06 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [17]:
# Revision sprint demo: rebuild weekly core diagnostics
np.random.seed(1506)
prices = load_market_prices(['SPY', 'QQQ', 'AAPL'], start='2018-01-01')
returns = prices.pct_change().dropna()

summary = pd.DataFrame({
    'annual_return': returns.mean() * 252,
    'annual_vol': returns.std() * np.sqrt(252),
    'max_drawdown': (prices / prices.cummax() - 1).min(),
})
summary['sharpe_proxy'] = (summary['annual_return'] - 0.03) / summary['annual_vol'].replace(0, np.nan)
summary = summary.sort_values('sharpe_proxy', ascending=False)

print('Revision diagnostics (best risk-adjusted first):')
summary.round(4)


Revision diagnostics (best risk-adjusted first):


,annual_return,annual_vol,max_drawdown,sharpe_proxy
Ticker,,,,
AAPL,0.277000,0.306000,-0.385200,0.807300
QQQ,0.205600,0.238800,-0.351200,0.735400
SPY,0.151700,0.193100,-0.337200,0.630400


## ReAct Verification Cell
Validate trade logic with a risk guardrail before reading the model quiz answers.

In [18]:
# ReAct-style verification: observe -> reason -> act -> verify
np.random.seed(10506)

observe_tickers = ['SPY', 'QQQ', 'TLT']
observe_prices = load_market_prices(observe_tickers, start='2020-01-01')
observe_returns = observe_prices.pct_change().dropna()

if observe_returns.empty:
    raise ValueError('No returns available for verification checks')

ann_vol = float(observe_returns['SPY'].std() * np.sqrt(252))
ann_ret = float((1 + observe_returns['SPY']).prod() ** (252 / len(observe_returns)) - 1)
sharpe_proxy = float((ann_ret - 0.03) / max(ann_vol, 1e-8))

# Risk-first deployment gate used in realistic interview responses.
guardrail = 'de-risk' if ann_vol > 0.30 else 'monitor'
decision = 'deploy-paper-trade' if sharpe_proxy > 0.40 and guardrail == 'monitor' else 'hold-and-review'

verification = {
    'topic': 'Revision Sprint',
    'week': 15,
    'day': 6,
    'observe_annual_return': ann_ret,
    'observe_annual_vol': ann_vol,
    'reason_sharpe_proxy': sharpe_proxy,
    'act_guardrail': guardrail,
    'verify_decision': decision,
}

verification


{'topic': 'Revision Sprint',
 'week': 15,
 'day': 6,
 'observe_annual_return': 0.14923378440213853,
 'observe_annual_vol': 0.20501289085899252,
 'reason_sharpe_proxy': 0.5815916448109953,
 'act_guardrail': 'monitor',
 'verify_decision': 'deploy-paper-trade'}

## Week 15 Day 06 Quiz

Topic: **Revision Sprint**

Real-world interview questions (answer first, then run the next cell for model answers):
1. PM question: Which formula from today's lesson directly drives a trade decision, and what does each symbol mean?
2. Risk question: Using one real ticker from today's example, what hard guardrail would you enforce before live deployment?
3. Communication question: In one minute, explain why this topic matters for production trading systems.

Scoring: full credit requires notation correctness, one numeric example, and one explicit risk control.

In [19]:
# Quiz model answers (auto-generated)
price_t_minus_1 = 121.000
price_t = 122.331
r_t = (price_t - price_t_minus_1) / price_t_minus_1
gross = 1 + r_t

print('Interview Question 1 (model answer):')
print('  I would use simple return to convert price moves into decision-ready percentages.')
print('  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)')
print('  P_(t-1):', price_t_minus_1)
print('  P_t    :', price_t)
print('  r_t    :', round(r_t, 6), '=>', f'{r_t*100:.2f}%')
print('  1+r_t  :', round(gross, 6))

print('\nInterview Question 2 (model answer):')
print('  For a real ticker like SPY, I would enforce a volatility guardrail: reduce position size')
print('  when realized volatility rises above regime threshold, then re-validate the signal logic.')

print('\nInterview Question 3 (model answer):')
print('  Topic:', 'Revision Sprint')
print('  This matters because production systems need reproducible metrics, explicit risk controls,')
print('  and clear decision rules that survive stressed market regimes.')

print('\nNumeric verification:')
print('  simple_return_expected =', 0.011000)
print('  gross_return_expected  =', 1.011000)


Interview Question 1 (model answer):
  I would use simple return to convert price moves into decision-ready percentages.
  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)
  P_(t-1): 121.0
  P_t    : 122.331
  r_t    : 0.011 => 1.10%
  1+r_t  : 1.011

Interview Question 2 (model answer):
  For a real ticker like SPY, I would enforce a volatility guardrail: reduce position size
  when realized volatility rises above regime threshold, then re-validate the signal logic.

Interview Question 3 (model answer):
  Topic: Revision Sprint
  This matters because production systems need reproducible metrics, explicit risk controls,
  and clear decision rules that survive stressed market regimes.

Numeric verification:
  simple_return_expected = 0.011
  gross_return_expected  = 1.011


# Week 15 Day 07: Portfolio Project

## Study Duration
- Planned effort: 2 hours

## Continuity and Handoff
- Previous checkpoint: Week 15 Day 06: Revision Sprint
- Previous lesson file: content/week-15/day-06.md
- Today's deliverable: Options risk intuition notebook
- Next handoff target: Week 16 Day 01: Risk framework architecture
- Next lesson file: content/week-16/day-01.md

## Project Title
Options risk intuition notebook

## Problem Statement
Build an options analytics notebook covering payoffs, Greeks, and hedging outcomes.

## Data Sources
- Synthetic option chain
- Underlying paths
- Volatility scenarios

## Implementation Steps
1. Implement payoff analytics
2. Estimate and visualize Greeks
3. Run hedging simulation
4. Compare strategy scenarios
5. Summarize practical risk insights

## Evaluation Metrics
- Hedge error
- Scenario PnL spread
- Risk clarity
- Interpretability

## Deliverables
- Notebook or script output
- One-page summary memo
- Tracker update with completion and lessons learned


## Week 15 Day 07 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [20]:
# Project day demo: mini portfolio report with trade recommendation
np.random.seed(1507)
assets = ['SPY', 'QQQ', 'TLT', 'GLD']
prices = load_market_prices(assets, start='2019-01-01')
returns = prices.pct_change().dropna()

annual_return = returns.mean() * 252
annual_vol = returns.std() * np.sqrt(252)
score = (annual_return - 0.03) / annual_vol.replace(0, np.nan)

report = pd.DataFrame({
    'annual_return': annual_return,
    'annual_vol': annual_vol,
    'sharpe_proxy': score,
}).sort_values('sharpe_proxy', ascending=False)

top_asset = report.index[0]
print('Project summary:')
print(report.round(4))
print(f"\nSuggested focus asset for follow-up research: {top_asset}")


Project summary:
        annual_return  annual_vol  sharpe_proxy
Ticker                                         
GLD          0.194100    0.172700      0.950000
QQQ          0.232200    0.240200      0.841900
SPY          0.177800    0.196000      0.754000
TLT         -0.005100    0.162600     -0.215600

Suggested focus asset for follow-up research: GLD


## ReAct Verification Cell
Validate trade logic with a risk guardrail before reading the model quiz answers.

In [21]:
# ReAct-style verification: observe -> reason -> act -> verify
np.random.seed(10507)

observe_tickers = ['SPY', 'QQQ', 'TLT']
observe_prices = load_market_prices(observe_tickers, start='2020-01-01')
observe_returns = observe_prices.pct_change().dropna()

if observe_returns.empty:
    raise ValueError('No returns available for verification checks')

ann_vol = float(observe_returns['SPY'].std() * np.sqrt(252))
ann_ret = float((1 + observe_returns['SPY']).prod() ** (252 / len(observe_returns)) - 1)
sharpe_proxy = float((ann_ret - 0.03) / max(ann_vol, 1e-8))

# Risk-first deployment gate used in realistic interview responses.
guardrail = 'de-risk' if ann_vol > 0.30 else 'monitor'
decision = 'deploy-paper-trade' if sharpe_proxy > 0.40 and guardrail == 'monitor' else 'hold-and-review'

verification = {
    'topic': 'Portfolio Project',
    'week': 15,
    'day': 7,
    'observe_annual_return': ann_ret,
    'observe_annual_vol': ann_vol,
    'reason_sharpe_proxy': sharpe_proxy,
    'act_guardrail': guardrail,
    'verify_decision': decision,
}

verification


{'topic': 'Portfolio Project',
 'week': 15,
 'day': 7,
 'observe_annual_return': 0.14923378440213853,
 'observe_annual_vol': 0.20501289085899252,
 'reason_sharpe_proxy': 0.5815916448109953,
 'act_guardrail': 'monitor',
 'verify_decision': 'deploy-paper-trade'}

## Week 15 Day 07 Quiz

Topic: **Portfolio Project**

Real-world interview questions (answer first, then run the next cell for model answers):
1. PM question: Which formula from today's lesson directly drives a trade decision, and what does each symbol mean?
2. Risk question: Using one real ticker from today's example, what hard guardrail would you enforce before live deployment?
3. Communication question: In one minute, explain why this topic matters for production trading systems.

Scoring: full credit requires notation correctness, one numeric example, and one explicit risk control.

In [22]:
# Quiz model answers (auto-generated)
price_t_minus_1 = 122.000
price_t = 123.403
r_t = (price_t - price_t_minus_1) / price_t_minus_1
gross = 1 + r_t

print('Interview Question 1 (model answer):')
print('  I would use simple return to convert price moves into decision-ready percentages.')
print('  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)')
print('  P_(t-1):', price_t_minus_1)
print('  P_t    :', price_t)
print('  r_t    :', round(r_t, 6), '=>', f'{r_t*100:.2f}%')
print('  1+r_t  :', round(gross, 6))

print('\nInterview Question 2 (model answer):')
print('  For a real ticker like SPY, I would enforce a volatility guardrail: reduce position size')
print('  when realized volatility rises above regime threshold, then re-validate the signal logic.')

print('\nInterview Question 3 (model answer):')
print('  Topic:', 'Portfolio Project')
print('  This matters because production systems need reproducible metrics, explicit risk controls,')
print('  and clear decision rules that survive stressed market regimes.')

print('\nNumeric verification:')
print('  simple_return_expected =', 0.011500)
print('  gross_return_expected  =', 1.011500)


Interview Question 1 (model answer):
  I would use simple return to convert price moves into decision-ready percentages.
  Formula: r_t = (P_t - P_(t-1)) / P_(t-1)
  P_(t-1): 122.0
  P_t    : 123.403
  r_t    : 0.0115 => 1.15%
  1+r_t  : 1.0115

Interview Question 2 (model answer):
  For a real ticker like SPY, I would enforce a volatility guardrail: reduce position size
  when realized volatility rises above regime threshold, then re-validate the signal logic.

Interview Question 3 (model answer):
  Topic: Portfolio Project
  This matters because production systems need reproducible metrics, explicit risk controls,
  and clear decision rules that survive stressed market regimes.

Numeric verification:
  simple_return_expected = 0.0115
  gross_return_expected  = 1.0115
